In [0]:
# 1. Definição dos caminhos na Landing Zone simulada
base_path = "file:/Workspace/Shared/healthcare_project/"
landing_zone_path = f"{base_path}landing_zone/"
checkpoint_path = f"{base_path}checkpoints/"
schema_path = f"{base_path}schemas/"

# Limpeza de execuções anteriores (para reutilização do exercício)
dbutils.fs.rm(landing_zone_path, recurse=True)
dbutils.fs.rm(checkpoint_path, recurse=True)
dbutils.fs.rm(schema_path, recurse=True)

# 2. Criação do primeiro lote de dados (Lote 1: Dados padrão de pacientes)
batch_1_data = """encounter_id,patient_id,encounter_type,blood_pressure,heart_rate,timestamp
ENC_001,PAT_1001,Emergency,120/80,72,2026-08-11T10:00:00
ENC_002,PAT_1002,Routine,115/75,68,2026-08-11T10:15:00
ENC_003,PAT_1003,Inpatient,135/90,85,2026-08-11T10:30:00
"""

# Gravando o arquivo CSV simulando um drop de arquivo via sistema hospitalar
dbutils.fs.put(f"{landing_zone_path}batch_1.csv", batch_1_data, overwrite=True)
print("Lote 1 disponibilizado na Landing Zone com sucesso!")

In [0]:
from pyspark.sql.functions import col, current_timestamp

df_auto_loader = (spark.readStream
                   .format("cloudFiles")
                   .option("cloudFiles.format", "csv")
                   .option("cloudFiles.schemaLocation", schema_path)
                   .option("header", "true")
                   .option("cloudFiles.inferColumnTypes", "true")
                   .option("cloudFiles.schemaNameConflictMode", "addNewColumns")
                   .load(landing_zone_path)
                   .withColumn("ingest_timestamp", current_timestamp())
                  )

query = (df_auto_loader.writeStream
         .format("delta")
         .outputMode("append")
         .option("checkpointLocation", checkpoint_path)
         .trigger(availableNow=True)
         .table("bronze_patient_encounters")
        )

query.awaitTermination()
print("Processamento do Auto Loader concluído com sucesso!")

In [0]:
# 1. Buscando o diretório dinâmico liberado pelo cluster
warehouse_dir = spark.conf.get("spark.sql.warehouse.dir")
landing_zone_path = f"{warehouse_dir}/fde_project/landing_zone/"
checkpoint_path = f"{warehouse_dir}/fde_project/checkpoints/"
schema_path = f"{warehouse_dir}/fde_project/schemas/"

# Limpeza de tabelas e checkpoints de tentativas anteriores
spark.sql("DROP TABLE IF EXISTS bronze_patient_encounters_fde")
try:
    dbutils.fs.rm(f"{warehouse_dir}/fde_project/", recurse=True)
except:
    pass # Ignora se o dbutils estiver 100% bloqueado até para deletar

# 2. Criando o Lote 1 de forma segura via DataFrame nativo
data_batch_1 = [
    ("ENC_001", "PAT_1001", "Emergency", "120/80", 72, "2026-08-11T10:00:00"),
    ("ENC_002", "PAT_1002", "Routine", "115/75", 68, "2026-08-11T10:15:00"),
    ("ENC_003", "PAT_1003", "Inpatient", "135/90", 85, "2026-08-11T10:30:00")
]
columns_1 = ["encounter_id", "patient_id", "encounter_type", "blood_pressure", "heart_rate", "timestamp"]

df_batch_1 = spark.createDataFrame(data_batch_1, columns_1)

# O Spark grava o arquivo usando as próprias permissões internas, burlando a restrição
df_batch_1.write.format("csv").option("header", "true").mode("overwrite").save(landing_zone_path)

print(f"Lote 1 salvo com sucesso no diretório seguro do motor: {landing_zone_path}")